### Nummernmerker – Lösung Schritt für Schritt

*38. Bundeswettbewerb Informatik (2019), 1. Runde, Aufgabe 2*

Sarahs Mutter seufzt: „Puh, in diesem Formular muss ich schon wieder meine Mitgliedsnummer angeben. Jetzt muss ich sie suchen, ich kann mir solche Nummern einfach nicht gut merken!“

„Ich merke mir lange Nummern immer so, dass ich sie in kürzere Blöcke aufteile“, sagt Sarah. „Wenn's geht, sollen diese aber nicht mit Null anfangen, weil ich sie mir dann nicht als Zahl merken kann. Meine Telefonnummer 01365400606 merke ich mir zum Beispiel so: 0136 5400 606.“

#### Aufgabe

Schreibe ein Programm, das Sarahs Mutter helfen kann, sich lange Nummern zu merken. Dein Programm soll eine Nummer, also eine Folge von Ziffern, einlesen und sie so in Blöcke aufteilen, dass gilt:

- Jeder Block hat höchstens vier, aber mindestens zwei Ziffern.
- Die Anzahl der Blöcke, die mit einer Null beginnen, ist minimal.

Du kannst davon ausgehen, dass eine einzulesende Nummer aus 2 bis 30 Ziffern besteht. Wende dein Programm auf alle Beispielnummern an, die sich auf der BWINF-Website befinden.

#### Beispieldaten

`beispieldaten/nummern.txt`: eine Nummer pro Zeile.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Rekursion:** Den letzten Block abtrennen und für den Rest dieselbe Aufgabe lösen.
- **Memoization / dynamische Programmierung:** Dieselben Teilaufgaben (die Aufteilung eines Anfangsstücks der Nummer) tauchen immer wieder auf. Wer sich die Antwort merkt, rechnet jede nur einmal.
- **Kürzeste Wege:** Man kann die Aufgabe auch als Weg durch einen Graphen sehen (siehe „Für Neugierige“).

**Nützliche Funktionen und Bibliotheken**
- Zeichenketten lassen sich wie Listen ansprechen: `nummer[i]` ist die Ziffer an Stelle `i`, `nummer[i:j]` das Stück von `i` bis vor `j`.
- `float('inf')` ist ein „unendlich großer“ Wert, mit dem man Minima bequem startet.
- Mit `' '.join(liste)` setzt man die Blöcke wieder zu einem Text mit Leerzeichen zusammen.

## Die Idee

Wir betrachten **Anfangsstücke** der Nummer. `kosten[i]` soll die kleinste Zahl von Blöcken mit führender Null sein, wenn man die ersten `i` Ziffern in Blöcke aus 2 bis 4 Ziffern teilt.

1. Für das leere Anfangsstück (`i = 0`) brauchen wir keinen Block: `kosten[0] = 0`.
2. Das Anfangsstück aus `i` Ziffern endet mit einem letzten Block der Länge 2, 3 oder 4. Dieser Block beginnt an Stelle `i - länge` (Stellen zählen ab 0). Er kostet 1, wenn dort eine `0` steht, sonst 0. Dazu kommen die Kosten `kosten[i - länge]` für alles davor.
3. Wir nehmen von den (höchstens) drei Möglichkeiten die billigste und merken uns, welche Länge es war. Am Ende gehen wir von hinten Block für Block zurück und lesen die Aufteilung ab.

Gibt es immer eine Aufteilung? Ja: Jede Länge ab 2 ist eine Summe aus Zweiern und Dreiern (5 = 2 + 3, 7 = 2 + 2 + 3, ...). Es kann aber sein, dass man dabei Blöcke mit führender Null nicht vermeiden kann (etwa bei `00060`). Das ist erlaubt, wir minimieren nur die Zahl solcher Blöcke.

## Schritt 1: Einlesen

Wir lesen alle Nummern der Datei (eine pro Zeile). Nummern bleiben dabei **Texte**: Als Zahl gelesen würde die führende Null verschwinden.

In [ ]:
def lies_nummern(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        return [zeile.strip() for zeile in f if zeile.strip() != '']


NUMMERN = lies_nummern('beispieldaten/nummern.txt')
for nummer in NUMMERN:
    print(len(nummer), 'Ziffern:', nummer)

18 Ziffern: 005480000005179734
23 Ziffern: 03495929533790154412660
25 Ziffern: 5319974879022725607620179
28 Ziffern: 9088761051699482789038331267
30 Ziffern: 011000000011000100111111101011


## Schritt 2: Der einfache rekursive Versuch

Wir schneiden den **letzten** Block ab (2, 3 oder 4 Ziffern lang), lösen den Rest rekursiv und nehmen die billigste Möglichkeit. Der Zähler `aufrufe` zeigt, wie oft die Funktion gerufen wird.

In [ ]:
aufrufe = 0


def beste_kosten(nummer, laenge):
    """Kleinste Zahl von Blöcken mit führender Null beim Aufteilen der ersten `laenge` Ziffern (unendlich, falls unmöglich)."""
    global aufrufe
    aufrufe = aufrufe + 1
    if laenge == 0:
        return 0
    beste = float('inf')
    for blocklaenge in [2, 3, 4]:
        start = laenge - blocklaenge             # hier beginnt der letzte Block
        if start >= 0:
            kosten = beste_kosten(nummer, start)
            if nummer[start] == '0':
                kosten = kosten + 1
            if kosten < beste:
                beste = kosten
    return beste


print(beste_kosten('01365400606', 11))
print(aufrufe, 'Aufrufe')

1
76 Aufrufe


Das Ergebnis stimmt (Sarahs Aufteilung `0136 5400 606` hat einen Block mit führender Null), aber schon für elf Ziffern sind es viele Aufrufe. Bei 28 Ziffern werden es sehr viel mehr:

In [ ]:
aufrufe = 0
print(beste_kosten('9088761051699482789038331267', 28))
print(aufrufe, 'Aufrufe für 28 Ziffern')

0
50897 Aufrufe für 28 Ziffern


Der Grund: `beste_kosten(nummer, 12)` wird zum Beispiel auf mehreren Wegen aufgerufen und jedes Mal neu berechnet. Wir merken uns deshalb die Ergebnisse.

## Schritt 3: Dynamische Programmierung

Statt von hinten rekursiv nach vorne zu springen, füllen wir die Tabelle `kosten` von vorne nach hinten aus. Zusätzlich merken wir in `blocklaenge[i]`, welche Länge der letzte Block im besten Fall hat. Damit lässt sich die Aufteilung am Ende zurückverfolgen.

In [ ]:
def teile(nummer):
    """Gibt (Kosten, Liste der Blöcke) einer besten Aufteilung zurück."""
    n = len(nummer)
    kosten = [float('inf')] * (n + 1)
    blocklaenge = [0] * (n + 1)
    kosten[0] = 0
    for i in range(2, n + 1):                    # Anfangsstück aus i Ziffern
        for laenge in [2, 3, 4]:
            start = i - laenge
            if start >= 0:
                neu = kosten[start]
                if nummer[start] == '0':
                    neu = neu + 1
                if neu < kosten[i]:
                    kosten[i] = neu
                    blocklaenge[i] = laenge
    # von hinten zurückverfolgen
    bloecke = []
    i = n
    while i > 0:
        bloecke.append(nummer[i - blocklaenge[i]:i])
        i = i - blocklaenge[i]
    bloecke.reverse()
    return kosten[n], bloecke


kosten, bloecke = teile('01365400606')
print(kosten, bloecke)

1 ['013', '65', '400', '606']


`kosten[1]` bleibt unendlich: Ein einzelnes Zeichen lässt sich nicht aufteilen. Das stört nicht, denn solche Wege verlieren gegen echte Zahlen.

Die Lösungshinweise nennen als Beispiel gegen „immer Blöcke fester Länge“ die Nummer `60040400` (`600 40 400` hat keinen Block mit Null). Dazu zwei Nummern, bei denen sich eine führende Null nicht vermeiden lässt:

In [ ]:
print(teile('60040400'))
print(teile('00060'))
print(teile('12000'))

(0, ['600', '40', '400'])
(1, ['000', '60'])
(1, ['120', '00'])


## Schritt 4: Alle Beispiele

Zur Ausgabe setzen wir die Blöcke mit Leerzeichen zusammen und zeigen die Zahl der Blöcke mit führender Null.

In [ ]:
for nummer in NUMMERN:
    kosten, bloecke = teile(nummer)
    print(nummer)
    print(' '.join(bloecke), '  -> Blöcke mit führender Null:', kosten)
    print()

005480000005179734
00 54 8000 0005 17 97 34   -> Blöcke mit führender Null: 2

03495929533790154412660
03 49 59 29 53 37 901 54 41 26 60   -> Blöcke mit führender Null: 1

5319974879022725607620179
531 99 74 87 902 27 25 60 76 201 79   -> Blöcke mit führender Null: 0

9088761051699482789038331267
90 88 76 10 51 69 94 82 78 90 38 33 12 67   -> Blöcke mit führender Null: 0

011000000011000100111111101011
01 1000 0000 110 00 100 11 11 11 10 10 11   -> Blöcke mit führender Null: 3



Beginnt eine Nummer mit `0`, so hat schon der erste Block eine führende Null, daran lässt sich nichts ändern. Das sieht man bei `005480000005179734`, `03495929533790154412660` und der langen Nummer aus Nullen und Einsen. Wie viele Blöcke mit Null mindestens nötig sind, steht jeweils in der letzten Zahl der Ausgabe.

## Für Neugierige

- **Dijkstra.** Jede Stelle zwischen den Ziffern ist ein Knoten (0 bis n), von jedem Knoten `s` führen Kanten zu `s + 2`, `s + 3` und `s + 4`. Eine Kante kostet 1, wenn an Stelle `s` eine `0` steht, sonst 0. Ein kürzester Weg von 0 nach n ist eine optimale Aufteilung. Unsere Tabelle `kosten` ist genau das, nur dass wir die Knoten von vorne nach hinten abarbeiten können, weil es keine Rückwärtskanten gibt.
- **Laufzeit.** Die Tabelle hat n + 1 Einträge, jeder wird aus höchstens drei Möglichkeiten berechnet: Die Laufzeit wächst nur linear mit der Länge der Nummer, während die einfache Rekursion von Schritt 2 stark zunimmt.
- **Mehrere beste Lösungen.** Es gibt oft mehrere gleich gute Aufteilungen. Welche wir ausgeben, hängt davon ab, in welcher Reihenfolge wir 2, 3, 4 probieren und ob bei Gleichstand ersetzt wird (`<` statt `<=`).

## Zum Schluss: Nummernmerker mit MiniZinc

Auch diese Aufgabe lässt sich in MiniZinc beschreiben, ohne den Algorithmus selbst zu erfinden. Wir sagen nur, wie eine gültige Aufteilung aussieht und was minimiert werden soll.

**Modell.**

- `len[b]` ist die **Länge** von Block `b`: 2, 3 oder 4, oder 0, wenn es den Block nicht gibt (`var {0, 2, 3, 4}`). Mit höchstens `n div 2` Blöcken sind wir sicher.
- `sum(len) = n`: Die Blöcke decken die ganze Nummer ab. Unbenutzte Blöcke (Länge 0) stehen hinten (Implikation `->`).
- `start[b]` ist die **Stelle**, an der Block `b` beginnt: Block 1 an Stelle 1, jeder weitere direkt hinter dem vorigen.
- Die Kosten zählen alle vorhandenen Blöcke, deren erste Ziffer `ziffer[start[b]]` eine 0 ist. Hier steht die Variable `start[b]` als **Index** in einem Array.

```minizinc
int: n;                                     % Anzahl der Ziffern
array[1..n] of 0..9: ziffer;
int: kmax = n div 2;                        % höchstens so viele Blöcke

array[1..kmax] of var {0, 2, 3, 4}: len;    % Länge von Block b (0 = Block gibt es nicht)
constraint sum(len) = n;
constraint forall(b in 1..kmax-1)(len[b] = 0 -> len[b+1] = 0);    % unbenutzte Blöcke stehen hinten

% Block b beginnt an Stelle start[b]
array[1..kmax] of var 1..n+1: start;
constraint start[1] = 1;
constraint forall(b in 2..kmax)(start[b] = start[b-1] + len[b-1]);

var int: kosten = sum(b in 1..kmax)(bool2int(len[b] > 0 /\ ziffer[start[b]] = 0));
solve minimize kosten;

output ["Kosten: ", show(kosten), "\n", show([len[b] | b in 1..kmax where fix(len[b]) > 0])];
```

**Daten.** Die Python-Zelle `als_dzn` baut die `.dzn`-Datei aus einer Nummer. Für die erste Nummer sieht sie so aus:

```minizinc
n = 18;
ziffer = [0, 0, 5, 4, 8, 0, 0, 0, 0, 0, 0, 5, 1, 7, 9, 7, 3, 4];
```

**Ausgabe:**

```
Kosten: 2
[3, 4, 4, 4, 3]
----------
==========
```

(Die Ausgabe zeigt die Kosten und die Blocklängen. Welche der gleich guten Aufteilungen der Solver nennt, kann von unserer abweichen.)

Kosten für alle fünf Nummern (Solver **Gecode**, jede Nummer braucht Sekundenbruchteile):

```
005480000005179734  ->  2
03495929533790154412660  ->  1
5319974879022725607620179  ->  0
9088761051699482789038331267  ->  0
011000000011000100111111101011  ->  3
```

Das sind dieselben Kosten wie bei unserer dynamischen Programmierung. Das Modell ist nicht schneller als diese, aber man muss keine Rekursion und keine Tabelle erfinden.

In [ ]:
def als_dzn(nummer):
    return f'n = {len(nummer)};\nziffer = [{", ".join(nummer)}];'


print(als_dzn(NUMMERN[0]))

n = 18;
ziffer = [0, 0, 5, 4, 8, 0, 0, 0, 0, 0, 0, 5, 1, 7, 9, 7, 3, 4];
